# KOBIS 영화 정보 수집
셀을 **순서대로** 실행하세요.  
중단해도 `data/kobis_progress.json`에 저장되어 재실행 시 이어서 진행됩니다.

In [20]:
import json, time, re, requests, pandas as pd
from pathlib import Path

# ── API 키 설정 ──────────────────────────────────────
API_KEYS = [
    "7eccea85dd0b53c5a9646f1e012297c6",
    "6c473256f22b9b3d9e50983e6d322725",
    "470e9ac05da21ed137381f0b878e4ea7",
    "d584d161685528e42a5d34cc6eb82c47",
    "da7a44d27226f75c18e8588170e7be10",
    "ced897d260349ad1fd4bfdfac8c644bf",
    "524283f348b55f75461e7b726425187b",
    "f3761b1a552eba9ab32b62dfb2fadfd4",
    "880da37637c3f986d6eb4e3243aa658b",
    "bacdd7462fa288520c44b7668d6335be",
    "7921cebeda88d15034069a7bf396cb96",
    "41cfdaf9f543e7d5625fa56b2cdc064c",
]
DAILY_LIMIT   = 3500
REQUEST_DELAY = 0.25

SEARCH_URL  = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieList.json"
DETAIL_URL  = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieInfo.json"

DATA_DIR      = Path("../data")
MOVIE_MASTER  = DATA_DIR / "Movie_Master.csv"
PROGRESS_FILE = DATA_DIR / "kobis_progress.json"
OUTPUT_FILE   = DATA_DIR / "Movie_Master_kobis.csv"

print("설정 완료")


설정 완료


In [21]:
class KeyManager:
    def __init__(self, keys, daily_limit):
        self.keys = keys
        self.daily_limit = daily_limit
        self.usage = {k: 0 for k in keys}

    def get(self):
        for k in self.keys:
            if self.usage[k] < self.daily_limit:
                return k
        return None

    def use(self, key):
        self.usage[key] += 1

    def remaining(self):
        return sum(self.daily_limit - v for v in self.usage.values())


def make_queries(title):
    queries = []
    raw = title.strip()
    queries.append(raw)
    no_bracket = re.sub(r'\[.*?\]', '', raw).strip()
    if no_bracket and no_bracket != raw:
        queries.append(no_bracket)
    no_year = re.sub(r'\(\d{4}\)', '', no_bracket or raw).strip()
    if no_year and no_year not in queries:
        queries.append(no_year)
    no_paren = re.sub(r'[\(\[(\u3010][^)\])\u3011]*[\)\])\u3011]', '', raw).strip()
    if no_paren and no_paren not in queries:
        queries.append(no_paren)
    # 콜론 앞부분만 추출 (예: "기생충:기생충" → "기생충")
    if ':' in (no_paren or raw):
        before_colon = (no_paren or raw).split(':')[0].strip()
        if before_colon and before_colon not in queries:
            queries.append(before_colon)
    base = no_paren or no_year or raw
    if len(base) > 6:
        queries.append(base[:6])
    if len(base) > 4:
        queries.append(base[:4])
    seen = []
    for q in queries:
        if q and q not in seen:
            seen.append(q)
    return seen


def search_api(query, km):
    key = km.get()
    if not key:
        raise RuntimeError("모든 API 키 일일 한도 초과")
    params = {"key": key, "movieNm": query, "itemPerPage": 10}
    km.use(key)
    try:
        resp = requests.get(SEARCH_URL, params=params, timeout=10)
        resp.raise_for_status()
        result = resp.json().get("movieListResult", {})
        return result.get("movieList", []), int(result.get("totCnt", 0))
    except Exception as e:
        print(f"  [검색 오류] {query}: {e}")
        return [], 0


def fetch_detail(movie_cd, km):
    key = km.get()
    if not key:
        raise RuntimeError("모든 API 키 일일 한도 초과")
    params = {"key": key, "movieCd": movie_cd}
    km.use(key)
    try:
        resp = requests.get(DETAIL_URL, params=params, timeout=10)
        resp.raise_for_status()
        return resp.json().get("movieInfoResult", {}).get("movieInfo", {})
    except Exception as e:
        print(f"  [상세 오류] {movie_cd}: {e}")
        return {}


def title_similarity(a, b):
    from difflib import SequenceMatcher
    a2 = re.sub(r'\s', '', a)
    b2 = re.sub(r'\s', '', b)
    return SequenceMatcher(None, a2, b2).ratio()


def is_good_match(original_title, result_title, similarity_threshold=0.6):
    clean_orig = re.sub(r'[\(\[（【][^)\]）】]*[\)\]）】]', '', original_title).strip()
    if ':' in clean_orig:
        clean_orig = clean_orig.split(':')[0].strip()

    clean_q = re.sub(r'\s', '', clean_orig)
    clean_r = re.sub(r'\s', '', result_title)

    # 1~3자: API 결과도 정확히 같은 글자수여야 통과
    if len(clean_q) <= 3:
        return clean_q == clean_r

    # 4자 이상: 기존 유사도 방식
    sim = title_similarity(clean_orig, result_title)
    return sim >= similarity_threshold

def pick_best(candidates, title, release_month, similarity_threshold=0.6):
    if not candidates:
        return None
    valid = [c for c in candidates if is_good_match(title, c.get("movieNm", ""), similarity_threshold)]
    if not valid:
        return None
    if len(valid) == 1:
        return valid[0]
    clean_q = re.sub(r'[\(\[（【][^)\]）】]*[\)\]）】]', '', title).strip()
    exact = [c for c in valid if c.get("movieNm", "") == clean_q]
    pool = exact if len(exact) >= 1 else valid
    if release_month and str(release_month).isdigit():
        wavve_year = int(str(release_month)[:4])
        def year_dist(c):
            od = c.get("openDt", "") or ""
            return abs(int(od[:4]) - wavve_year) if len(od) >= 4 and od[:4].isdigit() else 9999
        pool.sort(key=year_dist)
    return pool[0]


def parse_detail(info):
    if not info:
        return {}
    def join(lst, key):
        return "|".join(x.get(key, "") for x in lst if x.get(key))
    return {
        "MovieCd"      : info.get("movieCd", ""),
        "movieNm(api)" : info.get("movieNm", ""),
        "movieNmEn"    : info.get("movieNmEn", ""),
        "showTm"       : info.get("showTm", ""),
        "prdtYear"     : info.get("prdtYear", ""),
        "openDt"       : info.get("openDt", ""),
        "typeNm"       : info.get("typeNm", ""),
        "nationNm"     : join(info.get("nations", []), "nationNm"),
        "genreNm"      : join(info.get("genres", []), "genreNm"),
        "directors"    : join(info.get("directors", []), "peopleNm"),
        "actors"       : join(info.get("actors", [])[:5], "peopleNm"),
        "watchGrade"   : info.get("audits", [{}])[0].get("watchGradeNm", "") if info.get("audits") else "",
    }


def _save_progress(progress):
    with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
        json.dump(progress, f, ensure_ascii=False, indent=2)


def _build_output(mv, progress):
    rows = []
    for _, row in mv.iterrows():
        mid = str(row["MOVIE_ID"])
        info = progress["results"].get(mid, {})
        merged = row.to_dict()
        merged.update(info)
        rows.append(merged)
    pd.DataFrame(rows).to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")


print("함수 정의 완료")

함수 정의 완료


In [23]:
mv = pd.read_csv(MOVIE_MASTER)
print(f"Movie_Master 로드: {len(mv)}개 영화")

# 크롤링 필요 영화.txt가 있으면 해당 영화만 대상으로 (전체 재크롤링)
NEEDED_FILE = Path("../크롤링 필요 영화.txt")
if NEEDED_FILE.exists():
    needed = set(l.strip() for l in NEEDED_FILE.read_text(encoding="utf-8-sig").splitlines() if l.strip())
    mv = mv[mv["TITLE"].isin(needed)].reset_index(drop=True)
    print(f"크롤링 필요 영화.txt 로드: {len(needed)}개 → Movie_Master 매칭: {len(mv)}개")
    needed_mode = True
else:
    needed_mode = False

if PROGRESS_FILE.exists():
    with open(PROGRESS_FILE, encoding="utf-8") as f:
        progress = json.load(f)
    print(f"이전 진행 상황 로드: {len(progress['results'])}개 완료")
else:
    progress = {"results": {}, "failed": [], "not_found": []}

done_ids = set() if needed_mode else set(mid for mid, v in progress["results"].items() if v.get("MovieCd"))
km       = KeyManager(API_KEYS, DAILY_LIMIT)
total    = len(mv)
todo     = mv[~mv["MOVIE_ID"].astype(str).isin(done_ids)]

print(f"남은 작업: {len(todo)}개 / 전체 {total}개")
print(f"가용 API 요청 수: {km.remaining()}건")

for idx, row in todo.iterrows():
    movie_id      = str(row["MOVIE_ID"])
    title         = str(row["TITLE"])
    release_month = row.get("RELEASE_MONTH", None)

    # 제목 괄호 안 (YYYY) 있으면 그걸 연도 기준으로 우선 사용, 없으면 release_month 사용
    m = re.search(r'\((\d{4})\)', title)
    year_for_search = int(m.group(1)) if m else release_month

    done_cnt = len(progress["results"]) + len(progress["not_found"])
    if done_cnt % 100 == 0:
        print(f"[{done_cnt}/{total}] 남은 키 요청: {km.remaining()}건")

    if km.remaining() <= 0:
        print("모든 키 한도 소진. 내일 재실행하세요.")
        break

    queries = make_queries(title)
    found, matched_query, tot_cnt = None, None, 0

    for q in queries:
        time.sleep(REQUEST_DELAY)
        candidates, tot = search_api(q, km)
        if tot > 0:
            tot_cnt = tot
        best = pick_best(candidates, title, year_for_search, similarity_threshold=0.8)
        if best:
            found, matched_query = best, q
            break

    if not found:
        print(f"  [미발견] {title}")
        progress["not_found"].append({"MOVIE_ID": movie_id, "TITLE": title})
        progress["results"][movie_id] = {"totCnt": tot_cnt}
    else:
        time.sleep(REQUEST_DELAY)
        detail = fetch_detail(found["movieCd"], km)
        parsed = parse_detail(detail)
        parsed["totCnt"] = tot_cnt
        if matched_query != title:
            print(f"  [fallback] {title} -> {matched_query} -> {found.get('movieNm', '')}")
        progress["results"][movie_id] = parsed

    if (len(progress["results"]) + len(progress["not_found"])) % 100 == 0:
        _save_progress(progress)

_save_progress(progress)
# 크롤링 필요 영화만 별도 저장
_needed_rows = []
for _, _r in mv.iterrows():
    _mid = str(_r["MOVIE_ID"])
    _info = progress["results"].get(_mid, {})
    _merged = _r.to_dict()
    _merged.update(_info)
    _needed_rows.append(_merged)
pd.DataFrame(_needed_rows).to_csv(DATA_DIR / "크롤링_필요_영화_kobis.csv", index=False, encoding="utf-8-sig")
print(f"새 파일 저장: {DATA_DIR}/크롤링_필요_영화_kobis.csv")
found_cnt = sum(1 for v in progress["results"].values() if v.get("MovieCd"))
print(f"완료! CSV 저장: {OUTPUT_FILE}")
print(f"발견: {found_cnt}개 / 미발견: {len(progress['not_found'])}개")

Movie_Master 로드: 14018개 영화
크롤링 필요 영화.txt 로드: 755개 → Movie_Master 매칭: 755개
이전 진행 상황 로드: 14018개 완료
남은 작업: 755개 / 전체 755개
가용 API 요청 수: 42000건
  [미발견] 벌룬
  [fallback] 철권:블러드벤젠스 -> 철권 -> 철권
  [fallback] 얼굴없는보스:못다한이야기[감독판] -> 얼굴없는보스 -> 얼굴없는 보스
  [fallback] 에일리언어브덕션(2014) -> 에일리언어브덕션 -> 에일리언 애브덕션
  [미발견] 복수(1970)
  [미발견] 0.0mhz
  [미발견] 7년:그들이없는언론
  [fallback] 캠퍼스레전드(1998) -> 캠퍼스레전드 -> 캠퍼스 레전드
  [미발견] 캣츠뮤지컬특별공연
  [미발견] 독고리와인드(2018)[상]
  [fallback] 호날두:세상은그의발아래에 -> 호날두 -> 호날두
  [fallback] 소녀검객아즈미대혈전(2003) -> 소녀검객아즈미대혈전 -> 소녀 검객 아즈미 대혈전
  [fallback] 하이스쿨뮤지컬:졸업반 -> 하이스쿨뮤지컬 -> 하이 스쿨 뮤지컬 2
  [미발견] 비틀스:에잇데이즈어위크투어링이어즈
  [fallback] 오션월드[2D] -> 오션월드 -> 오션월드 3D
  [미발견] 할힌골전투:일본군최후의날
  [fallback] 종횡사해(1991) -> 종횡사해 -> 종횡사해
  [fallback] 데드풀2:순한맛 -> 데드풀2 -> 데드풀 2
  [미발견] 일론머스크:리얼아이언맨
  [fallback] 스타워즈2:클론의습격 -> 스타워즈 -> 스타워즈
  [미발견] 고사:피의중간고사(2008)
  [fallback] 자칼(2017) -> 자칼 -> 자칼
  [미발견] <소울>나의이웃[별도편성]
  [미발견] 파워레인져스:더비기닝
  [fallback] 킹아더:원탁의기사 -> 킹아더 -> 킹 아더
  [미발견] 헝거게임:캣칭파이어
  [미발견] 로닌(1998)
  [fallbac

In [ ]:
import json, pandas as pd

p = json.load(open("data/kobis_progress.json", encoding="utf-8"))
mv = pd.read_csv("data/Movie_Master.csv")

rows = []
for _, row in mv.iterrows():
    mid = str(row["MOVIE_ID"])
    info = p["results"].get(mid, {})
    merged = row.to_dict()
    merged.update(info)
    rows.append(merged)

pd.DataFrame(rows).to_csv("data/Movie_Master_kobis_temp.csv", index=False, encoding="utf-8-sig")
print("저장 완료: data/Movie_Master_kobis_temp.csv")


저장 완료: data/Movie_Master_kobis_temp.csv


In [ ]:
import requests

total_ok = 0
for i, key in enumerate(API_KEYS):
    try:
        resp = requests.get(
            SEARCH_URL,
            params={"key": key, "movieNm": "기생충", "itemPerPage": 1},
            timeout=10
        )
        result = resp.json().get("movieListResult", {})
        if "totCnt" in result:
            print(f"키{i+1}: ✓ 정상")
            total_ok += 1
        else:
            print(f"키{i+1}: ✗ 한도 초과")
    except Exception as e:
        print(f"키{i+1}: ✗ 오류 - {e}")

print(f"\n사용 가능: {total_ok} / {len(API_KEYS)}개")
print(f"가용 요청 수 (추정): {total_ok * DAILY_LIMIT:,}건")

키1: ✓ 정상
키2: ✓ 정상
키3: ✓ 정상
키4: ✓ 정상
키5: ✓ 정상
키6: ✓ 정상
키7: ✓ 정상
키8: ✓ 정상
키9: ✓ 정상
키10: ✓ 정상
키11: ✓ 정상
키12: ✓ 정상

사용 가능: 12 / 12개
가용 요청 수 (추정): 36,000건


In [ ]:
import json, time, re

# ── 설정 ──────────────────────────────────────────────
LOW_THRESHOLD = 0.55

# ── 로드 ──────────────────────────────────────────────
p  = json.load(open("data/kobis_progress.json", encoding="utf-8"))
mv = pd.read_csv(MOVIE_MASTER)
id2row = {str(r["MOVIE_ID"]): r for _, r in mv.iterrows()}

failed_ids = [mid for mid, v in p["results"].items() if not v.get("MovieCd")]
print(f"재시도 대상: {len(failed_ids)}개  (유사도 기준: {LOW_THRESHOLD})")

# ── 연도 기준으로만 선택 (pick_best 미사용) ───────────
def pick_by_year_only(candidates, year_for_search):
    if not candidates:
        return None
    if year_for_search and str(year_for_search).isdigit():
        yr = int(str(year_for_search)[:4])
        def dist(c):
            od = c.get("openDt", "") or ""
            return abs(int(od[:4]) - yr) if len(od) >= 4 and od[:4].isdigit() else 9999
        return min(candidates, key=dist)
    return candidates[0]

# ── 재시도 ──────────────────────────────────────────────
km = KeyManager(API_KEYS, DAILY_LIMIT)
found_count = 0

for i, mid in enumerate(failed_ids):
    row = id2row.get(mid)
    if row is None:
        continue
    title         = str(row["TITLE"])
    release_month = row.get("RELEASE_MONTH", None)

    m = re.search(r'\((\d{4})\)', title)
    year_for_search = int(m.group(1)) if m else release_month

    if km.remaining() <= 0:
        print("모든 키 한도 소진.")
        break

    queries = make_queries(title)
    found, tot_cnt = None, 0

    for q in queries:
        time.sleep(REQUEST_DELAY)
        candidates, tot = search_api(q, km)
        if tot > 0:
            tot_cnt = tot
        valid = [c for c in candidates
                 if is_good_match(title, c.get("movieNm", ""), LOW_THRESHOLD)]
        if valid:
            found = pick_by_year_only(valid, year_for_search)
            break

    if not found:
        p["results"][mid]["totCnt"] = tot_cnt
        print(f"  [실패] {title}")
    else:
        time.sleep(REQUEST_DELAY)
        detail = fetch_detail(found["movieCd"], km)
        parsed = parse_detail(detail)
        parsed["totCnt"] = tot_cnt
        p["results"][mid] = parsed
        found_count += 1
        print(f"  [발견] {title} → {found.get('movieNm', '')}")

    if (i + 1) % 50 == 0:
        _save_progress(p)

_save_progress(p)
_build_output(mv, p)

still_failed = [mid for mid, v in p["results"].items() if not v.get("MovieCd")]
print("")
print(f"새로 발견: {found_count}개 / 여전히 미발견: {len(still_failed)}개")
print(f"전체 발견: {sum(1 for v in p['results'].values() if v.get('MovieCd'))}개 / {len(mv)}개")


재시도 대상: 620개  (유사도 기준: 0.55)
  [실패] 뤽베송의마지막전투
  [실패] 디즈니네이쳐:베어
  [실패] 매드헌터
  [실패] 화장실어디에요?
  [실패] 뜨는뉴욕브루클린
  [실패] 궁합
  [실패] 로스타임라이프후반전2
  [실패] 7년:그들이없는언론
  [실패] 캣츠뮤지컬특별공연
  [실패] 쓰리:메모리즈
  [실패] 길(1954)
  [실패] 고보이즈:마지막잎새프로젝트
  [실패] 스트립클럽
  [발견] 백만달러의사랑 → 백만달러와 시체
  [실패] 데블
  [실패] 이다
  [발견] 주성치의심사관 → 주성치의 가유희사 97


KeyboardInterrupt: 

## wavve_notfound.txt → KOBIS 재검색 (유사도 0.8)

In [ ]:
# ── 설정 ──────────────────────────────────────────────
NOTFOUND_TXT    = Path("wavve_notfound.txt")
OUTPUT_FOUND    = Path("data/wavve_notfound_kobis.csv")
OUTPUT_NOTFOUND = Path("data/wavve_notfound_still.txt")
SIMILARITY_THR  = 0.8

# ── wavve_notfound.txt 로드 ────────────────────────────
titles = [line.strip() for line in NOTFOUND_TXT.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"검색 대상: {len(titles)}개 제목")

# ── API 키 매니저 초기화 ────────────────────────────────
km = KeyManager(API_KEYS, DAILY_LIMIT)
print(f"가용 API 요청 수: {km.remaining():,}건")


In [ ]:
found_rows  = []   # 매칭 성공 결과
still_not   = []   # 여전히 미발견

for i, title in enumerate(titles, start=1):
    if km.remaining() <= 0:
        print("모든 API 키 한도 소진. 내일 재실행하세요.")
        break

    queries  = make_queries(title)
    best_hit = None
    tot_cnt  = 0

    for q in queries:
        time.sleep(REQUEST_DELAY)
        candidates, tot = search_api(q, km)
        if tot > 0:
            tot_cnt = tot

        # 유사도 0.8 기준으로 후보 필터링
        valid = [c for c in candidates
                 if is_good_match(title, c.get("movieNm", ""), SIMILARITY_THR)]
        if valid:
            best_hit = valid[0]   # 첫 번째 유효 후보 선택
            break

    if best_hit:
        time.sleep(REQUEST_DELAY)
        detail = fetch_detail(best_hit["movieCd"], km)
        parsed = parse_detail(detail)
        parsed["wavve_title"] = title          # 원본 wavve 제목도 보존
        found_rows.append(parsed)
        print(f"  [{i:>4}] ✓ {title[:25]:<25} → {best_hit.get('movieNm','')}")
    else:
        still_not.append(title)
        if i % 50 == 0:
            print(f"  [{i:>4}] ... (미발견 누적: {len(still_not)}개)")

# ── 결과 저장 ─────────────────────────────────────────
if found_rows:
    pd.DataFrame(found_rows).to_csv(OUTPUT_FOUND, index=False, encoding="utf-8-sig")
    print(f"\n매칭 성공 저장: {OUTPUT_FOUND}  ({len(found_rows)}개)")

OUTPUT_NOTFOUND.write_text("\n".join(still_not), encoding="utf-8")
print(f"여전히 미발견 저장: {OUTPUT_NOTFOUND}  ({len(still_not)}개)")
print(f"\n결과 요약  |  성공: {len(found_rows)}  /  실패: {len(still_not)}  /  전체: {len(titles)}")


NameError: name 'titles' is not defined